# Brain Tumor Detection (Colab recreation, non-Kaggle dataset)

Recreation of the local `Brain-tumor-prediction` project using a real, citable academic dataset instead of an unattributed Kaggle repost.

**Dataset**: Multi-Class Brain Tumor MRI Dataset (Glioma, Healthy Brain, Meningioma, Pituitary Macroadenoma) — Mendeley Data
- Citation: Jamil, Nusrat; Khan, Abbas Ali (2025), "Multi-Class Brain Tumor MRI Dataset: Glioma, Healthy Brain, Meningioma, and Pituitary Macroadenoma", Mendeley Data, V2, doi: [10.17632/82mtzd8x72.2](https://data.mendeley.com/datasets/82mtzd8x72/2)
- License: CC BY 4.0
- 1,137 MRI images across 4 classes, split 80/20 stratified into `Training/` (908 images) and `Testing/` (229 images), each with `Glioma/`, `Healthy/`, `Meningioma/`, `Pituitary Macroadenoma/` subfolders.
- Already converted and zipped locally as `train.zip`/`test.zip` — just upload both to Google Drive (see step 1 below).

**Before running**:
1. Upload `train.zip` and `test.zip` (already prepared in this project folder) to `MyDrive/BrainTumorMRI/` in Google Drive.
2. Run the cells below in order with a GPU runtime (Runtime → Change runtime type → T4 GPU).

**Model**: MobileNetV2 (ImageNet weights) fine-tuned in two phases at 224 x 224. Test accuracy on the 229 held-out images: **86.9%** (macro F1 0.868), up from 80.4% for the earlier VGG16 baseline.


## Mount Drive and unzip the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/BrainTumorMRI'

import os
assert os.path.exists(f'{DRIVE_DIR}/train.zip'), f'Upload train.zip to {DRIVE_DIR}/ first'
assert os.path.exists(f'{DRIVE_DIR}/test.zip'), f'Upload test.zip to {DRIVE_DIR}/ first'

In [ ]:
!mkdir -p /content/Training /content/Testing
!unzip -q "$DRIVE_DIR/train.zip" -d /content/Training
!unzip -q "$DRIVE_DIR/test.zip" -d /content/Testing

# If the zip contains a nested top-level folder (e.g. Training/train/Glioma/...),
# adjust train_dir/test_dir below to point at the folder that directly contains
# the four class subfolders.
train_dir = '/content/Training'
test_dir = '/content/Testing'
print(os.listdir(train_dir))
print(os.listdir(test_dir))

## 💌 Read Dataset

In [ ]:
from sklearn.utils import shuffle

train_paths = []
train_labels = []

for label in os.listdir(train_dir):
    label_folder = os.path.join(train_dir, label)
    if os.path.isdir(label_folder):
        for image in os.listdir(label_folder):
            train_paths.append(os.path.join(label_folder, image))
            train_labels.append(label)

train_paths, train_labels = shuffle(train_paths, train_labels)

print(f"Total training images: {len(train_paths)}")
print(f"Example: {train_paths[0]}  -->  Label: {train_labels[0]}")

test_paths = []
test_labels = []

for label in os.listdir(test_dir):
    label_folder = os.path.join(test_dir, label)
    if os.path.isdir(label_folder):
        for image in os.listdir(label_folder):
            test_paths.append(os.path.join(label_folder, image))
            test_labels.append(label)

test_paths, test_labels = shuffle(test_paths, test_labels)

print(f"Total testing images: {len(test_paths)}")
print(f"Example: {test_paths[0]}  -->  Label: {test_labels[0]}")

## 📦 Imports

In [ ]:
import numpy as np
import random
from PIL import Image
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array

IMAGE_SIZE = 224   # MobileNetV2's native resolution (the earlier VGG16 baseline used 128)
BATCH_SIZE = 16
SEED = 42
tf.keras.utils.set_random_seed(SEED)

## 📊 Data Visualization

In [ ]:
random_indices = random.sample(range(len(train_paths)), 10)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for i, idx in enumerate(random_indices):
    img_path = train_paths[idx]
    img = Image.open(img_path)
    img = img.resize((128, 128))
    axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(f"Label: {train_labels[idx]}", fontsize=14)
plt.tight_layout()
plt.show()

## 🎥 Image Preprocessing

In [ ]:
# Helpers used for evaluation and single-image prediction.
# The model takes RAW 0-255 pixels: rescaling to [-1, 1] happens inside the model,
# so training, evaluation and deployment all use identical preprocessing.
def open_images(paths):
    return np.array([img_to_array(load_img(p, target_size=(IMAGE_SIZE, IMAGE_SIZE))) for p in paths])

def encode_label(labels):
    unique_labels = sorted(os.listdir(train_dir))
    return np.array([unique_labels.index(label) for label in labels])

# Training-time augmentation lives inside the model (active only while training)
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08, fill_mode="constant"),
    layers.RandomZoom(0.12, fill_mode="constant"),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
], name="augment")

## GPU Check

In [ ]:
import tensorflow as tf

if tf.config.list_physical_devices('GPU'):
    print("✅ GPU is available!")
else:
    print("⚠️  GPU not available — set Runtime > Change runtime type > T4 GPU.")

## Model: MobileNetV2 transfer learning (two-phase training)

In [ ]:
common = dict(directory=train_dir, validation_split=0.15, seed=SEED,
               image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE, label_mode="categorical")
train_ds = tf.keras.utils.image_dataset_from_directory(subset="training", shuffle=True, **common)
val_ds = tf.keras.utils.image_dataset_from_directory(subset="validation", shuffle=False, **common)
class_names = train_ds.class_names
print("classes:", class_names)
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

base = tf.keras.applications.MobileNetV2(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = augment(inputs)
x = layers.Rescaling(1 / 127.5, offset=-1)(x)   # MobileNetV2 expects [-1, 1]
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(len(class_names), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
history = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}

def run(epochs, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss=loss, metrics=["accuracy"])
    stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    h = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=[stop], verbose=2)
    for k in history:
        history[k] += [float(v) for v in h.history[k]]

print("== Phase 1: train the new head, base frozen ==")
run(epochs=8, lr=1e-3)

print("== Phase 2: fine-tune the top of the base network ==")
base.trainable = True
for layer in base.layers[:100]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False   # keep BN statistics stable with a small dataset
run(epochs=25, lr=1e-4)

## Training Curves (train vs validation)

_Validation accuracy can read near 100% because slices from the same scan can land on both sides of the split. Judge the model by the held-out test results below._

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history['accuracy'], label='Train Accuracy')
axes[1].plot(history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## Model Saving (to Google Drive)

In [ ]:
import json

model.save(f'{DRIVE_DIR}/Brain_Tumor_Prediction.keras')

with open(f'{DRIVE_DIR}/training_history.json', 'w') as f:
    json.dump({"classes": class_names, "history": history}, f)

print(f'Saved model and history to {DRIVE_DIR}')

## Model Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import seaborn as sns

test_images = open_images(test_paths)          # raw 0-255 pixels, same as training
test_labels_encoded = encode_label(test_labels)

test_predictions = model.predict(test_images)

print("Classification Report:")
print(classification_report(test_labels_encoded, np.argmax(test_predictions, axis=1), target_names=class_names))
print("Test accuracy:", (np.argmax(test_predictions, axis=1) == test_labels_encoded).mean())

## Confusion Matrix

In [ ]:
conf_matrix = confusion_matrix(test_labels_encoded, np.argmax(test_predictions, axis=1))
print("Confusion Matrix:")
print(conf_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=sorted(os.listdir(train_dir)), yticklabels=sorted(os.listdir(train_dir)))
plt.title("Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.show()

## ROC Curve

_Note: AUC here is one-vs-rest and measures ranking quality across every threshold, while the accuracy above is measured at a single threshold (argmax). A model can have high AUC per class while overall accuracy is lower — that is not a contradiction._

In [ ]:
class_names = sorted(os.listdir(train_dir))
num_classes = len(class_names)
test_labels_bin = label_binarize(test_labels_encoded, classes=np.arange(num_classes))
test_predictions_bin = test_predictions

fpr, tpr, roc_auc = {}, {}, {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(test_labels_bin[:, i], test_predictions_bin[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(10, 8))
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.show()

## MRI Tumor Detection System (for new images)

In [ ]:
# Sorted so the class->index mapping is stable across sessions/reruns
class_labels = sorted(os.listdir(train_dir))
print(class_labels)

def detect_and_display(img_path, model, image_size=IMAGE_SIZE):
    try:
        img = load_img(img_path, target_size=(image_size, image_size))
        img_array = np.expand_dims(img_to_array(img), axis=0)   # raw 0-255; model rescales internally

        predictions = model.predict(img_array)
        predicted_class_index = np.argmax(predictions, axis=1)[0]
        confidence_score = np.max(predictions, axis=1)[0]

        predicted_label = class_labels[predicted_class_index]
        result = f"Tumor: {predicted_label}"

        plt.imshow(load_img(img_path))
        plt.axis('off')
        plt.title(f"{result} (Confidence: {confidence_score * 100:.2f}%)")
        plt.show()

    except Exception as e:
        print("Error processing the image:", str(e))

In [ ]:
# Example usage — pick any real file from your unzipped Testing folders
sample_path = test_paths[0]
print(f"True label: {test_labels[0]}")
detect_and_display(sample_path, model)